In [1]:
from func import *

# Read data

In [2]:
train_df = pd.read_csv("processed5/train_data.csv")
test_df = pd.read_csv("processed5/test_data.csv")

train_df["logClosePrice"] = np.log1p(train_df["ClosePrice"])
train_df.drop(columns=["ClosePrice"], inplace=True)
test_df["logClosePrice"] = np.log1p(test_df["ClosePrice"])
test_df.drop(columns=["ClosePrice"], inplace=True)

# Elastic Net

In [3]:
from sklearn.linear_model import ElasticNet

In [4]:

# Tuning
en_param_grid = param_grid = {
    "model__alpha": np.logspace(-4, 2, 25),
    "model__l1_ratio": np.linspace(0.05, 0.95, 19),
}

res = grid_tune_with_make_model_pipeline(
    train_df=train_df,
    target_col="logClosePrice",
    model=ElasticNet(max_iter=20000),
    param_grid=param_grid,
    col_drop_list=["ClosePrice"],
    card_threshold=20,
    scoring="r2",
    cv=5
)

print(res["best_score"], res["best_params"])

Fitting 5 folds for each of 475 candidates, totalling 2375 fits
0.8627505332713833 {'model__alpha': 0.01778279410038923, 'model__l1_ratio': 0.05}


In [9]:
alpha = 0.01778279410038923
l_1_ratio = 0.05
model = ElasticNet(alpha=alpha,
                   l1_ratio=l_1_ratio)

elastic_net_result = fit_predict(
    train_df=train_df,
    test_df=test_df,
    model=model,
    col_drop_list=["ClosePrice"],
    target_col="logClosePrice",
    card_threshold=20
)
el_metrics = elastic_net_result["Metrics"]
print(el_metrics)

{'R2(log)': 0.8631442453920551, 'R2': 0.7470882296920329, 'MAPE': 16.248031888703224, 'MdAPE': 11.78838144577255, 'RMSE': 450125.8388503317, 'MAE': 205125.37304462388, 'Bias(mean residual)': -70140.18323704242, 'APE_95pct': 46.17924558617598, 'APE_99pct': 78.4421981297489, 'APE_max': 232.7074705189007, 'Train_R2(log)': 0.8992553084461341, 'Test_R2(log)': 0.8631442453920551, 'R2_gap': 0.03611106305407896}


# Random Forest

In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

In [6]:
X_train = train_df.drop(columns=["logClosePrice"], axis=1)

num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

nunique = X_train[cat_cols].nunique(dropna=False)

low_card_cols = nunique[(nunique <= 20)].index.tolist()
high_card_cols = nunique[(nunique >= 20)].index.tolist()


In [3]:

rf_param_dist = {
    "model__n_estimators": randint(300, 1000),
    "model__max_depth": [None] + list(range(10, 60, 10)),
    "model__max_features": ["sqrt", "log2", 0.3, 0.5, 0.7],
    "model__min_samples_leaf": randint(1, 15),
    "model__min_samples_split": randint(2, 20)
}

model = make_model_pipeline(model=RandomForestRegressor(
    n_jobs=1,
    random_state=42),
    num_cols=num_cols,
    low_card_cols=low_card_cols,
    high_card_cols=high_card_cols
)


rs = RandomizedSearchCV(
    model,
    param_distributions=rf_param_dist,
    n_iter=40,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

rs.fit(train_df.drop("logClosePrice", axis=1), train_df["logClosePrice"])
print(rs.best_params_)

{'model__max_depth': 40, 'model__max_features': 'log2', 'model__min_samples_leaf': 8, 'model__min_samples_split': 13, 'model__n_estimators': 713}


In [16]:
n_estimators = 713
max_depth = 40
max_features = 'log2'
min_samples_leaf = 8
min_samples_split = 13

rf_model = RandomForestRegressor(n_estimators=n_estimators,
                                 max_depth=max_depth,
                                 max_features=max_features,
                                 min_samples_leaf=min_samples_leaf,
                                 min_samples_split=min_samples_split)

rf_result = fit_predict(
    train_df=train_df,
    test_df=test_df,
    model=rf_model,
    col_drop_list=["ClosePrice"],
    target_col="logClosePrice",
    card_threshold=20
)

rf_metrics = rf_result["Metrics"]
print(rf_metrics)

{'R2(log)': 0.7078687540468314, 'R2': 0.48194030576630686, 'MAPE': 23.939160475487725, 'MdAPE': 17.845857643963654, 'RMSE': 635441.4149342186, 'MAE': 301176.1640791882, 'Bias(mean residual)': -161738.5898028468, 'APE_95pct': 64.84903200263648, 'APE_99pct': 113.08048199798868, 'APE_max': 291.28577274560973, 'Train_R2(log)': 0.9696370881452737, 'Test_R2(log)': 0.7078687540468314, 'R2_gap': 0.26176833409844236}


# XGB

In [7]:
from xgboost import XGBRegressor
from scipy.stats import randint, uniform, loguniform

In [8]:

xgb_param_dist = {
    "model__max_depth": randint(3, 7),                    # 3–6
    "model__min_child_weight": randint(8, 31),           # 8–30
    "model__learning_rate": loguniform(0.01, 0.08),      # small LR
    "model__n_estimators": randint(400, 1400),
    "model__subsample": uniform(0.6, 0.3),               # 0.6–0.9
    "model__colsample_bytree": uniform(0.5, 0.4),        # 0.5–0.9
    "model__reg_lambda": loguniform(1.0, 50.0),          # strong L2
    "model__reg_alpha": loguniform(1e-4, 2.0),           # mild–moderate L1
}

model = make_model_numeric_only_pipeline(model=XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=1),
 num_cols=num_cols,
num_scaler=None)


xgb_rs = RandomizedSearchCV(
    model,
    param_distributions=xgb_param_dist,
    n_iter=40,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

xgb_rs.fit(train_df.drop("logClosePrice", axis=1), train_df["logClosePrice"])
print(xgb_rs.best_params_)

{'model__colsample_bytree': 0.7844598129752072, 'model__learning_rate': 0.05383345943137039, 'model__max_depth': 6, 'model__min_child_weight': 14, 'model__n_estimators': 1219, 'model__reg_alpha': 1.10973369349242, 'model__reg_lambda': 4.736558858366902, 'model__subsample': 0.755325405158244}


In [10]:
xgb_model = XGBRegressor(n_estimators=1219,
                         learning_rate=0.05383345943137039,
                         max_depth=6,
                         subsample=0.755325405158244,
                         colsample_bytree=0.7844598129752072,
                         min_child_weight=14,
                         reg_lambda=4.736558858366902,
                         reg_alpha=1.10973369349242)

# xgb_model = XGBRegressor(n_estimators=xgb_rs.best_params_["model__n_estimators"],
#                          learning_rate=xgb_rs.best_params_["model__learning_rate"],
#                          max_depth=xgb_rs.best_params_["model__max_depth"],
#                          subsample=xgb_rs.best_params_["model__subsample"],
#                          colsample_bytree=xgb_rs.best_params_["model__colsample_bytree"],
#                          min_child_weight=xgb_rs.best_params_["model__min_child_weight"],
#                          reg_lambda=xgb_rs.best_params_["model__reg_lambda"],
#                          reg_alpha=xgb_rs.best_params_["model__reg_alpha"]
#                          )

xgb_result = fit_predict(
    train_df=train_df,
    test_df=test_df,
    model=xgb_model,
    col_drop_list=["ClosePrice"],
    target_col="logClosePrice",
    card_threshold=20,
    num_scaler=None,
    num_only=True
)

xgb_metrics = xgb_result["Metrics"]
print(xgb_metrics)

{'R2(log)': 0.9224351015790591, 'R2': 0.8754771564087329, 'MAPE': 11.900525265072222, 'MdAPE': 8.266243101256984, 'RMSE': 312559.04972140153, 'MAE': 147339.54133504958, 'Bias(mean residual)': -18978.193168503843, 'APE_95pct': 35.14362449825699, 'APE_99pct': 60.446666712601086, 'APE_max': 193.78628138125578, 'Train_R2(log)': 0.953032013802761, 'Test_R2(log)': 0.9224351015790591, 'R2_gap': 0.030596912223701977}


# Stacking

In [11]:
from sklearn.ensemble import StackingRegressor


en_full = make_model_pipeline(
    model=ElasticNet(alpha=0.01778279410038923,
                     l1_ratio=0.05),
    num_cols=num_cols,
    low_card_cols=low_card_cols,
    high_card_cols=high_card_cols
)

xgb_num = make_model_numeric_only_pipeline(
    model=XGBRegressor(n_estimators=1219,
                       learning_rate=0.05383345943137039,
                       max_depth=6,
                       subsample=0.755325405158244,
                       colsample_bytree=0.7844598129752072,
                       min_child_weight=14,
                       reg_lambda=4.736558858366902,
                       reg_alpha=1.10973369349242),
    num_cols=num_cols,
    num_scaler=None
)

# ---------- Stack them ----------
stack = StackingRegressor(
    estimators=[
        ("en_full", en_full),
        ("xgb_num", xgb_num),
    ],
    final_estimator=XGBRegressor(
        max_depth=2,
        n_estimators=200,
        learning_rate=0.05
    ),
    cv=5,
    n_jobs=-1
)

# Fit on log target
y_train = train_df["logClosePrice"]
stack.fit(X_train, y_train)

# Predict log target
X_test = test_df.drop("logClosePrice", axis=1)
y_test = test_df["logClosePrice"]
y_train_pred = stack.predict(X_train)
y_pred = stack.predict(X_test)

stack_metrics = compute_metrics(y_test, y_pred, y_train, y_train_pred)

In [12]:
stack_metrics

{'R2(log)': 0.9253911611150452,
 'R2': 0.8798101284728096,
 'MAPE': 11.695055928457553,
 'MdAPE': 8.1907723621419,
 'RMSE': 306486.37080709654,
 'MAE': 143846.17354105416,
 'Bias(mean residual)': -14372.64268556503,
 'APE_95pct': 34.65268927129461,
 'APE_99pct': 59.84068509175975,
 'APE_max': 207.44608653776862,
 'Train_R2(log)': 0.953137998963144,
 'Test_R2(log)': 0.9253911611150452,
 'R2_gap': 0.027746837848098838}

# Residual model

In [13]:
residual_model = XGBRegressor(
    max_depth=3,
    learning_rate=0.03,
    n_estimators=400,
    subsample=0.8,
    colsample_bytree=0.8,
)

residual = y_train - y_train_pred

residual_model.fit(X_train[num_cols], residual)

pred1 = stack.predict(X_test)
pred2 = residual_model.predict(X_test[num_cols])

train_pred1 = stack.predict(X_train)
train_pred2 = residual_model.predict(X_train[num_cols])

y_residual_pred = pred1 + pred2
y_train_residual_pred = train_pred1 + train_pred2

In [14]:
residual_metrics = compute_metrics(y_test, y_residual_pred, y_train, y_train_residual_pred)

In [15]:
residual_metrics

{'R2(log)': 0.9267660256598371,
 'R2': 0.878406257727204,
 'MAPE': 11.465048646334994,
 'MdAPE': 7.956424855342325,
 'RMSE': 308746.75733005616,
 'MAE': 142516.8697422519,
 'Bias(mean residual)': -26294.528195562165,
 'APE_95pct': 34.002863751917566,
 'APE_99pct': 59.30172908343091,
 'APE_max': 198.91294353926568,
 'Train_R2(log)': 0.9550838075809993,
 'Test_R2(log)': 0.9267660256598371,
 'R2_gap': 0.02831778192116219}

# Summary

In [19]:
models_summary = {
    "Elastic Net": el_metrics,
    "Random Forest": rf_metrics,
    "XGBoost": xgb_metrics,
    "StackingRegressor": stack_metrics,
    "Residual Model": residual_metrics,
}

df = pd.DataFrame.from_dict(models_summary).transpose()
df

,R2(log),R2,MAPE,MdAPE,RMSE,MAE,Bias(mean residual),APE_95pct,APE_99pct,APE_max,Train_R2(log),Test_R2(log),R2_gap
Elastic Net,0.863144,0.747088,16.248032,11.788381,450125.838850,205125.373045,-70140.183237,46.179246,78.442198,232.707471,0.899255,0.863144,0.036111
Random Forest,0.707869,0.481940,23.939160,17.845858,635441.414934,301176.164079,-161738.589803,64.849032,113.080482,291.285773,0.969637,0.707869,0.261768
XGBoost,0.922435,0.875477,11.900525,8.266243,312559.049721,147339.541335,-18978.193169,35.143624,60.446667,193.786281,0.953032,0.922435,0.030597
StackingRegressor,0.925391,0.879810,11.695056,8.190772,306486.370807,143846.173541,-14372.642686,34.652689,59.840685,207.446087,0.953138,0.925391,0.027747
Residual Model,0.926766,0.878406,11.465049,7.956425,308746.757330,142516.869742,-26294.528196,34.002864,59.301729,198.912944,0.955084,0.926766,0.028318
